# 🎯 YOLOv1: You Only Look Once — Unified, Real-Time Object Detection

> **Kaynak Makale:** Redmon, J., Divvala, S., Girshick, R., & Farhadi, A. (2016).  
> *You Only Look Once: Unified, Real-Time Object Detection.* CVPR 2016.  
> 📄 [arXiv:1506.02640](https://arxiv.org/abs/1506.02640)

---

## 📚 İçindekiler
1. [Motivasyon & Arka Plan](#1)
2. [YOLO'nun Temel Fikri — Grid Tabanlı Tespit](#2)
3. [YOLO Mimarisi (Darknet)](#3)
4. [Loss Fonksiyonu — Detaylı Türetme](#4)
5. [Bounding Box Hesaplamaları & IoU](#5)
6. [Non-Maximum Suppression (NMS)](#6)
7. [Confidence Score & Prediction Decode](#7)
8. [Toy Dataset ile Uçtan Uca Demo](#8)
9. [Performans Analizi & Kısıtlamalar](#9)
10. [Özet & Sonraki Adımlar](#10)


<a id='1'></a>
## 1. 🧠 Motivasyon & Arka Plan

### YOLO Öncesi Nesne Tespiti

YOLO'dan önce nesne tespiti **iki aşamalı (two-stage)** bir süreçti:

| Yaklaşım | Yöntem | Sorun |
|----------|--------|-------|
| **Sliding Window** | Pencereyi kaydır, her bölgeye sınıflandırıcı uygula | Çok yavaş, binlerce değerlendirme |
| **R-CNN (2014)** | Selective Search → CNN → SVM | ~2 dakika/görüntü |
| **Fast R-CNN (2015)** | ROI Pooling, paylaşımlı CNN | Hâlâ ayrı region proposal aşaması |
| **Faster R-CNN (2015)** | Region Proposal Network (RPN) | ~0.2 sn/görüntü, hâlâ two-stage |

### YOLO'nun Devrim Niteliğindeki Yaklaşımı

> **"We reframe object detection as a single regression problem"**

YOLO, nesne tespitini **tek bir regresyon problemi** olarak yeniden tanımlar:  
Piksellerden → Bounding box koordinatları + Sınıf olasılıkları

**Sonuç:** 45 FPS (base), 155 FPS (Fast-YOLO) — gerçek zamanlı tespit mümkün!


<a id='2'></a>
## 2. 🔧 Kurulum & Import


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Stil ayarları
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['font.family'] = 'monospace'

print('✅ Tüm kütüphaneler yüklendi!')
print('📦 NumPy:', np.__version__)
import matplotlib; print('🎨 Matplotlib:', matplotlib.__version__)


## 2. 🗺️ YOLO'nun Temel Fikri — Grid Tabanlı Tespit

### Grid Cell Mantığı

YOLO, girdi görüntüsünü **S × S** bir ızgaraya böler (orijinal makalede S=7).  
Her grid cell sorumluluğunu şöyle taşır:

- **B adet** bounding box tahmini yapar (B=2 orijinal makalede)
- Her bounding box için **5 değer** tahmin eder: `(x, y, w, h, confidence)`
- **C adet** sınıf olasılığı tahmin eder (VOC dataset için C=20)

### Çıktı Tensörü Boyutu
$$\text{Output} = S \times S \times (B \times 5 + C) = 7 \times 7 \times (2 \times 5 + 20) = 7 \times 7 \times 30$$

### Önemli Kural
Bir nesnenin merkezi hangi grid cell'e düşüyorsa, o grid cell o nesneyi tespit etmekten **sorumludur**.


In [ ]:
def visualize_grid_concept():
    """YOLO'nun S×S grid mantığını görselleştirir"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor('#0d1117')
    
    S = 7  # Grid boyutu
    
    # Sol: Orijinal görüntü simülasyonu
    ax1 = axes[0]
    ax1.set_facecolor('#1a1a2e')
    img = np.random.rand(448, 448, 3) * 0.3 + 0.1
    # Basit nesne şekilleri ekle
    img[100:200, 80:180] = [0.2, 0.4, 0.8]   # Mavi kutu (araba)
    img[250:370, 290:400] = [0.8, 0.3, 0.2]  # Kırmızı kutu (kişi)
    ax1.imshow(img)
    ax1.set_title('📷 Orijinal Görüntü\n(448×448 px)', color='white', fontsize=12, pad=10)
    ax1.axis('off')
    
    # Orta: Grid overlay
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a2e')
    ax2.imshow(img)
    # Grid çiz
    for i in range(S+1):
        ax2.axhline(y=i*(448/S), color='#00ff88', linewidth=0.8, alpha=0.7)
        ax2.axvline(x=i*(448/S), color='#00ff88', linewidth=0.8, alpha=0.7)
    
    # Sorumlu hücreleri işaretle
    cell_size = 448 / S
    # Araba merkezi: (130, 150) → grid (2,1)
    car_cx, car_cy = 130, 150
    car_col, car_row = int(car_cx / cell_size), int(car_cy / cell_size)
    rect1 = patches.Rectangle((car_col*cell_size, car_row*cell_size), 
                                cell_size, cell_size,
                                linewidth=3, edgecolor='#ffff00', 
                                facecolor='yellow', alpha=0.3)
    ax2.add_patch(rect1)
    ax2.plot(car_cx, car_cy, 'y*', markersize=15)
    
    # Kişi merkezi: (345, 310) → grid (5,4)
    per_cx, per_cy = 345, 310
    per_col, per_row = int(per_cx / cell_size), int(per_cy / cell_size)
    rect2 = patches.Rectangle((per_col*cell_size, per_row*cell_size), 
                                cell_size, cell_size,
                                linewidth=3, edgecolor='#ff6b6b', 
                                facecolor='red', alpha=0.3)
    ax2.add_patch(rect2)
    ax2.plot(per_cx, per_cy, 'r*', markersize=15)
    
    ax2.set_title(f'🗺️ {S}×{S} Grid Bölümlemesi\n(Sarı=Araba sorumlusu, Kırmızı=Kişi sorumlusu)', 
                  color='white', fontsize=11, pad=10)
    ax2.axis('off')
    
    # Sağ: Output tensor şeması
    ax3 = axes[2]
    ax3.set_facecolor('#0d1117')
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 10)
    
    # Tensor kutusu
    tensor_colors = ['#e74c3c', '#e74c3c', '#3498db', '#3498db', '#2ecc71'*1]
    
    labels = [
        ('Box 1:\nx,y,w,h,conf', '#e74c3c', 0.5, 8),
        ('Box 2:\nx,y,w,h,conf', '#e67e22', 2.5, 8),
        ('C sınıf\nolasılığı\n(C=20)', '#3498db', 5.5, 8),
    ]
    
    for label, color, x, y in labels:
        box = FancyBboxPatch((x-0.9, y-1.5), 1.8, 3,
                              boxstyle='round,pad=0.1',
                              facecolor=color, alpha=0.7, edgecolor='white')
        ax3.add_patch(box)
        ax3.text(x, y, label, ha='center', va='center', 
                 color='white', fontsize=9, fontweight='bold')
    
    ax3.text(5, 5.5, '= 30 değer / hücre', ha='center', color='#00ff88', fontsize=13)
    ax3.text(5, 4.5, '7 × 7 × 30 tensor', ha='center', color='white', fontsize=14, fontweight='bold')
    ax3.text(5, 3.0, 'B×5 + C = 2×5+20 = 30', ha='center', color='#aaaaaa', fontsize=11)
    ax3.text(5, 9.5, '📦 YOLO Çıktı Tensörü', ha='center', color='white', fontsize=12, fontweight='bold')
    ax3.axis('off')
    
    plt.tight_layout(pad=2)
    plt.suptitle('YOLO v1: Grid Tabanlı Tespit Mimarisi', 
                 color='white', fontsize=15, fontweight='bold', y=1.02)
    plt.show()

visualize_grid_concept()


<a id='3'></a>
## 3. 🏗️ YOLO Mimarisi — Darknet

YOLO, GoogLeNet'ten ilham alan **Darknet** adlı özel bir CNN kullanır:

| Bölüm | Katmanlar | Açıklama |
|-------|-----------|----------|
| **Backbone** | Conv 1-20 | Feature extraction (ImageNet pretrained) |
| **Detection Head** | Conv 21-24 | Tespit için ek konvolüsyon |
| **Regressor** | FC1 + FC2 | 4096 → 7×7×30 |

```
Input: 448×448×3
   ↓  Conv(64, 7×7, s=2) → MaxPool(2×2, s=2)      → 112×112×64  → 56×56×64
   ↓  Conv(192, 3×3)     → MaxPool(2×2, s=2)       → 56×56×192  → 28×28×192
   ↓  Conv blocks (256,512) × 4 + Conv(512) → Pool → 14×14×512
   ↓  Conv blocks (512,1024)× 2 + Conv(1024)→ Pool → 7×7×1024
   ↓  Conv(1024, 3×3) × 2                          → 7×7×1024
   ↓  Flatten → FC(4096) → Dropout(0.5) → FC(1470) → reshape 7×7×30
```

**Aktivasyon:** Leaky ReLU (α=0.1) — son katman hariç tüm konvolüsyonlarda


In [ ]:
def draw_architecture():
    """YOLO Darknet mimarisini şematik olarak çizer"""
    fig, ax = plt.subplots(figsize=(20, 7))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    ax.set_xlim(0, 22)
    ax.set_ylim(0, 8)
    ax.axis('off')
    
    # Katman tanımları: (x, genişlik, yükseklik, renk, etiket, alt_etiket)
    layers = [
        (0.3,  0.6, 6.0, '#4ecdc4', 'Input\n448×448\n×3',    ''),
        (1.2,  0.5, 5.5, '#45b7d1', 'Conv\n7×7\ns=2',        '224×224\n×64'),
        (2.0,  0.4, 5.0, '#96ceb4', 'Pool\n2×2',             '112×112'),
        (2.7,  0.5, 4.5, '#45b7d1', 'Conv\n3×3',             '×192'),
        (3.5,  0.4, 4.2, '#96ceb4', 'Pool',                  '56×56'),
        (4.2,  0.7, 3.8, '#ffd93d', 'Conv\nBlocks\n×4',      '×256/512'),
        (5.2,  0.4, 3.5, '#96ceb4', 'Pool',                  '28×28'),
        (5.9,  0.7, 3.2, '#ffd93d', 'Conv\nBlocks\n×2',      '×1024'),
        (6.9,  0.4, 3.0, '#96ceb4', 'Pool',                  '14×14'),
        (7.6,  0.7, 2.8, '#45b7d1', 'Conv\n3×3\n×2',         '×1024'),
        (8.6,  0.7, 2.5, '#45b7d1', 'Conv\n3×3',             '7×7×1024'),
        (9.6,  0.7, 2.5, '#45b7d1', 'Conv\n3×3',             '7×7×1024'),
        (10.6, 0.8, 2.2, '#e74c3c', 'Flatten',               '50176'),
        (11.7, 0.8, 2.2, '#9b59b6', 'FC\n4096',              'Leaky ReLU'),
        (12.8, 0.6, 1.8, '#e67e22', 'Dropout\n0.5',          'train only'),
        (13.7, 0.8, 2.2, '#9b59b6', 'FC\n1470',              'Linear'),
        (15.0, 0.8, 3.5, '#2ecc71', 'Reshape\n7×7×30',       'Output\nTensor'),
    ]
    
    prev_x = None
    for i, (x, w, h, color, label, sublabel) in enumerate(layers):
        # Kutu çiz
        rect = FancyBboxPatch((x, (8-h)/2), w, h,
                               boxstyle='round,pad=0.05',
                               facecolor=color, alpha=0.85,
                               edgecolor='white', linewidth=1.2)
        ax.add_patch(rect)
        
        # Etiket
        ax.text(x + w/2, 4 + 0.15, label, ha='center', va='center',
                color='black', fontsize=7.5, fontweight='bold')
        if sublabel:
            ax.text(x + w/2, (8-h)/2 - 0.5, sublabel, ha='center', va='top',
                    color='#aaaaaa', fontsize=7)
        
        # Ok çiz
        if prev_x is not None:
            ax.annotate('', xy=(x, 4), xytext=(prev_x, 4),
                        arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5))
        prev_x = x + w
    
    # Bölüm etiketleri
    ax.text(5.5, 7.5, '◄── Backbone (Conv 1-20) ──►', ha='center', 
            color='#ffd93d', fontsize=11, style='italic')
    ax.text(9.5, 7.5, '◄─ Head ─►', ha='center', 
            color='#45b7d1', fontsize=11, style='italic')
    ax.text(13.5, 7.5, '◄── Regressor ──►', ha='center', 
            color='#9b59b6', fontsize=11, style='italic')
    
    ax.set_title('YOLOv1 — Darknet Mimarisi (24 Conv + 2 FC Katman)',
                 color='white', fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()

draw_architecture()


<a id='4'></a>
## 4. 📐 Loss Fonksiyonu — Detaylı Türetme

YOLO'nun kayıp fonksiyonu **5 bileşenden** oluşur:

$$\mathcal{L} = \underbrace{\lambda_{coord} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{obj} \left[(x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2\right]}_{\text{1. Merkez koordinat kaybı}}
+ \underbrace{\lambda_{coord} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{obj} \left[(\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2\right]}_{\text{2. Boyut kaybı (kareköklü)}}
$$

$$+ \underbrace{\sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{obj} (C_i - \hat{C}_i)^2}_{\text{3. Nesne var confidence kaybı}}
+ \underbrace{\lambda_{noobj} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{noobj} (C_i - \hat{C}_i)^2}_{\text{4. Nesne yok confidence kaybı}}
+ \underbrace{\sum_{i=0}^{S^2} \mathbb{1}_{i}^{obj} \sum_{c \in classes}(p_i(c) - \hat{p}_i(c))^2}_{\text{5. Sınıf kaybı}}
$$

### Neden Kareköklü Boyutlar?
Küçük kutulardaki hataları büyük kutulara kıyasla **eşit ağırlıklandırmak** için.  
Örneğin: 10px hata → büyük kutuda önemsiz, küçük kutuda çok önemli.

### Hiperparametreler
- $\lambda_{coord} = 5$ — koordinat hatasını büyüt (koordinatlar önemli!)
- $\lambda_{noobj} = 0.5$ — boş hücre güven hatasını küçült (çoğu hücre boş!)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def yolo_loss(predictions, targets, S=7, B=2, C=20, 
              lambda_coord=5.0, lambda_noobj=0.5):
    """
    YOLOv1 Loss Fonksiyonu — NumPy implementasyonu
    
    Args:
        predictions: (S, S, B*5 + C) — model çıktısı
        targets:     (S, S, B*5 + C) — ground truth
    Returns:
        Toplam kayıp ve bileşenler
    """
    loss_coord  = 0.0
    loss_size   = 0.0
    loss_obj    = 0.0
    loss_noobj  = 0.0
    loss_class  = 0.0

    for i in range(S):
        for j in range(S):
            for b in range(B):
                offset = b * 5
                # Tahminler
                pred_x, pred_y = predictions[i,j,offset],   predictions[i,j,offset+1]
                pred_w, pred_h = predictions[i,j,offset+2], predictions[i,j,offset+3]
                pred_conf      = predictions[i,j,offset+4]
                # Ground truth
                tgt_x,  tgt_y  = targets[i,j,offset],   targets[i,j,offset+1]
                tgt_w,  tgt_h  = targets[i,j,offset+2], targets[i,j,offset+3]
                tgt_conf       = targets[i,j,offset+4]
                
                obj_mask = tgt_conf > 0  # Nesne var mı?
                
                if obj_mask:
                    # 1. Koordinat kaybı
                    loss_coord += lambda_coord * ((pred_x - tgt_x)**2 + (pred_y - tgt_y)**2)
                    # 2. Boyut kaybı (kareköklü!)
                    loss_size  += lambda_coord * (
                        (np.sqrt(max(pred_w, 1e-6)) - np.sqrt(max(tgt_w, 1e-6)))**2 +
                        (np.sqrt(max(pred_h, 1e-6)) - np.sqrt(max(tgt_h, 1e-6)))**2
                    )
                    # 3. Confidence kaybı (nesne VAR)
                    loss_obj   += (pred_conf - tgt_conf)**2
                else:
                    # 4. Confidence kaybı (nesne YOK)
                    loss_noobj += lambda_noobj * (pred_conf - 0)**2
            
            # 5. Sınıf kaybı
            if targets[i,j,B*5-1] > 0:  # Hücrede nesne varsa
                pred_classes = predictions[i,j,B*5:]
                tgt_classes  = targets[i,j,B*5:]
                loss_class  += np.sum((pred_classes - tgt_classes)**2)

    total = loss_coord + loss_size + loss_obj + loss_noobj + loss_class
    return {
        'total':       total,
        'coord':       loss_coord,
        'size':        loss_size,
        'obj_conf':    loss_obj,
        'noobj_conf':  loss_noobj,
        'class':       loss_class
    }


# ── Kareköklü boyut neden önemli? Görselleştir ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

w = np.linspace(0.01, 1, 300)
delta = 0.1  # 0.1 birimlik sabit hata

# Gradyan karşılaştırması
grad_plain  = 2 * delta * np.ones_like(w)          # d(w-delta)^2 / dw = 2*delta (sabit)
grad_sqrt   = delta / np.sqrt(w + 1e-9)            # d(√w)^2/dw ≈ delta/√w (küçük w'de büyük)

ax1 = axes[0]
ax1.plot(w, grad_plain, color='#e74c3c', linewidth=2, label='Düz hata gradyanı (sabit)')
ax1.plot(w, grad_sqrt,  color='#2ecc71', linewidth=2, label='√w hata gradyanı (küçük kutuda büyük!)')
ax1.axvline(x=0.15, color='yellow', linestyle='--', alpha=0.5, label='Küçük kutu')
ax1.axvline(x=0.7,  color='cyan',   linestyle='--', alpha=0.5, label='Büyük kutu')
ax1.set_xlabel('Bounding Box Genişliği (w)', fontsize=11)
ax1.set_ylabel('Gradyan Büyüklüğü', fontsize=11)
ax1.set_title('Neden √w? — Küçük Kutu Hassasiyeti', color='white', fontsize=12)
ax1.legend(fontsize=9)
ax1.set_ylim(0, 3)

# Loss bileşen ağırlıkları
ax2 = axes[1]
components = ['Koordinat\n(×5)', 'Boyut\n(×5)', 'Conf (obj)', 'Conf (noobj)\n(×0.5)', 'Sınıf']
weights    = [5, 5, 1, 0.5, 1]
colors     = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']
bars = ax2.bar(components, weights, color=colors, edgecolor='white', linewidth=0.8)
for bar, w_val in zip(bars, weights):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'λ={w_val}', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax2.set_title('Loss Fonksiyonu Ağırlıkları (λ)', color='white', fontsize=12)
ax2.set_ylabel('Ağırlık Katsayısı', fontsize=11)
ax2.set_ylim(0, 6)

plt.tight_layout(pad=2)
plt.show()

# Test
np.random.seed(42)
dummy_pred = np.random.rand(7, 7, 30) * 0.5
dummy_tgt  = np.zeros((7, 7, 30))
dummy_tgt[3, 4, 4] = 1.0  # Grid (3,4)'te nesne var
dummy_tgt[3, 4, 0:4] = [0.5, 0.6, 0.3, 0.4]

result = yolo_loss(dummy_pred, dummy_tgt)
print('📊 YOLO Loss Bileşenleri (dummy data):')
for k, v in result.items():
    print(f'   {k:12s}: {v:.4f}')


<a id='5'></a>
## 5. 📦 Bounding Box Hesaplamaları & IoU

### Bounding Box Kodlaması

YOLO, koordinatları **grid cell'e göre normalize eder:**

| Koordinat | Anlamı | Aralık |
|-----------|--------|--------|
| `x, y` | Merkez, cell koordinatına göre offset | [0, 1] |
| `w, h` | Genişlik/yükseklik, görüntü boyutuna göre | [0, 1] |
| `confidence` | IoU × P(nesne var) | [0, 1] |

### Intersection over Union (IoU)

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Kesişim Alanı}}{\text{Birleşim Alanı}}$$

IoU eşiği genellikle **0.5** olarak kullanılır (Pascal VOC standardı).


In [ ]:
def compute_iou(box1, box2):
    """
    İki bounding box arasındaki IoU hesapla
    Her box: [x_center, y_center, width, height] formatında (0-1 normalize)
    """
    # Köşe koordinatlarına çevir
    b1_x1 = box1[0] - box1[2] / 2
    b1_y1 = box1[1] - box1[3] / 2
    b1_x2 = box1[0] + box1[2] / 2
    b1_y2 = box1[1] + box1[3] / 2

    b2_x1 = box2[0] - box2[2] / 2
    b2_y1 = box2[1] - box2[3] / 2
    b2_x2 = box2[0] + box2[2] / 2
    b2_y2 = box2[1] + box2[3] / 2

    # Kesişim
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Birleşim
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area

    return inter_area / (union_area + 1e-9)


def visualize_iou(box1, box2):
    """IoU hesabını görselleştirir"""
    iou = compute_iou(box1, box2)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')
    
    # Sol: Bounding box görselleştirmesi
    ax1 = axes[0]
    ax1.set_facecolor('#161b22')
    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
    ax1.set_aspect('equal')
    ax1.invert_yaxis()
    
    # Box 1 (prediction)
    b1 = patches.Rectangle(
        (box1[0]-box1[2]/2, box1[1]-box1[3]/2), box1[2], box1[3],
        linewidth=2.5, edgecolor='#e74c3c', facecolor='#e74c3c', alpha=0.3,
        label='Tahmin (Prediction)'
    )
    # Box 2 (ground truth)
    b2 = patches.Rectangle(
        (box2[0]-box2[2]/2, box2[1]-box2[3]/2), box2[2], box2[3],
        linewidth=2.5, edgecolor='#2ecc71', facecolor='#2ecc71', alpha=0.3,
        label='Ground Truth'
    )
    
    # Kesişim bölgesi
    ix1 = max(box1[0]-box1[2]/2, box2[0]-box2[2]/2)
    iy1 = max(box1[1]-box1[3]/2, box2[1]-box2[3]/2)
    ix2 = min(box1[0]+box1[2]/2, box2[0]+box2[2]/2)
    iy2 = min(box1[1]+box1[3]/2, box2[1]+box2[3]/2)
    
    if ix2 > ix1 and iy2 > iy1:
        inter = patches.Rectangle(
            (ix1, iy1), ix2-ix1, iy2-iy1,
            linewidth=0, facecolor='#f39c12', alpha=0.6, label='Kesişim (Intersection)'
        )
        ax1.add_patch(inter)
    
    ax1.add_patch(b1)
    ax1.add_patch(b2)
    
    # Merkezler
    ax1.plot(*box1[:2], 'r+', markersize=12, markeredgewidth=2)
    ax1.plot(*box2[:2], 'g+', markersize=12, markeredgewidth=2)
    
    iou_color = '#2ecc71' if iou > 0.5 else '#e74c3c'
    ax1.text(0.5, 0.08, f'IoU = {iou:.3f}', ha='center', transform=ax1.transAxes,
             fontsize=16, fontweight='bold', color=iou_color)
    ax1.text(0.5, 0.01, '✅ Eşleşme (IoU>0.5)' if iou>0.5 else '❌ Eşleşme Yok (IoU≤0.5)',
             ha='center', transform=ax1.transAxes, fontsize=11, color=iou_color)
    
    ax1.legend(loc='upper right', fontsize=9)
    ax1.set_title('Bounding Box IoU Hesabı', color='white', fontsize=12)
    ax1.grid(True, alpha=0.2)
    
    # Sağ: IoU değer aralıkları
    ax2 = axes[1]
    ax2.set_facecolor('#161b22')
    
    iou_vals = np.linspace(0, 1, 100)
    quality  = np.where(iou_vals < 0.3, 0,
                np.where(iou_vals < 0.5, 1,
                np.where(iou_vals < 0.75, 2, 3)))
    
    colors_q = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71']
    labels_q = ['Kötü (<0.3)', 'Zayıf (0.3-0.5)', 'İyi (0.5-0.75)', 'Mükemmel (>0.75)']
    
    for q in range(4):
        mask = quality == q
        ax2.fill_between(iou_vals[mask], 0, 1, alpha=0.3, color=colors_q[q], label=labels_q[q])
    
    ax2.axvline(x=0.5, color='white', linestyle='--', linewidth=2, label='VOC Eşiği (0.5)')
    ax2.axvline(x=iou,  color='yellow', linestyle='-',  linewidth=3, label=f'Hesaplanan IoU={iou:.3f}')
    
    ax2.set_xlabel('IoU Değeri', fontsize=11)
    ax2.set_title('IoU Kalite Aralıkları', color='white', fontsize=12)
    ax2.legend(fontsize=9)
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    
    plt.tight_layout(pad=2)
    plt.show()
    return iou


# Test
box_pred = [0.5, 0.5, 0.4, 0.4]  # Tahmin
box_gt   = [0.55, 0.55, 0.3, 0.3]  # Gerçek
iou = visualize_iou(box_pred, box_gt)
print(f'\n📊 IoU Değeri: {iou:.4f}')
print(f'   Sonuç: {"✅ Pozitif tespit (IoU > 0.5)" if iou > 0.5 else "❌ Negatif tespit (IoU ≤ 0.5)"}')


<a id='6'></a>
## 6. 🔧 Non-Maximum Suppression (NMS)

YOLO, aynı nesne için birden fazla bounding box üretebilir.  
**NMS**, tekrarlı tahminleri temizleyerek en iyi kutuyu seçer.

### NMS Algoritması (Adım Adım)

```
1. Tüm tahminleri confidence score'a göre azalan sıraya diz
2. En yüksek score'lu kutuyu seç → Keep listesine ekle
3. Kalan kutularla IoU hesapla
4. IoU > eşik (0.5) olan kutuları SİL (aynı nesneyi gösteriyorlar)
5. Kalan kutular bitene kadar 2-4'ü tekrarla
```

### Confidence Score Hesabı
$$\text{Confidence} = P(\text{nesne}) \times \text{IoU}^{truth}_{pred}$$

Test aşamasında:
$$\text{Class Confidence} = P(\text{class} | \text{nesne}) \times P(\text{nesne}) \times \text{IoU} = P(\text{class}) \times \text{IoU}$$


In [ ]:
def non_maximum_suppression(boxes, scores, iou_threshold=0.5, score_threshold=0.3):
    """
    Non-Maximum Suppression implementasyonu
    
    Args:
        boxes:          List of [x_center, y_center, w, h]
        scores:         Confidence scores
        iou_threshold:  Bu değerin üstündeki örtüşmeler silinir
        score_threshold: Bu değerin altındaki tahminler göz ardı edilir
    Returns:
        keep_indices: Seçilen kutuların indeksleri
    """
    if len(boxes) == 0:
        return []
    
    boxes  = np.array(boxes,  dtype=float)
    scores = np.array(scores, dtype=float)
    
    # Score eşiği filtresi
    valid = scores >= score_threshold
    boxes, scores = boxes[valid], scores[valid]
    orig_indices = np.where(valid)[0]
    
    # Score'a göre sırala (azalan)
    order = np.argsort(scores)[::-1]
    boxes, scores = boxes[order], scores[order]
    orig_indices  = orig_indices[order]
    
    keep = []
    remaining = list(range(len(boxes)))
    steps = []  # görselleştirme için
    
    while remaining:
        current = remaining[0]
        keep.append(orig_indices[current])
        
        new_remaining = []
        suppressed    = []
        
        for idx in remaining[1:]:
            iou = compute_iou(boxes[current], boxes[idx])
            if iou <= iou_threshold:
                new_remaining.append(idx)
            else:
                suppressed.append(idx)
        
        steps.append({
            'selected': current,
            'suppressed': suppressed,
            'kept': keep[:],
        })
        remaining = new_remaining
    
    return keep, steps


def visualize_nms():
    """NMS sürecini adım adım görselleştirir"""
    # Simüle edilmiş tahminler — aynı nesne etrafında çakışan kutular
    boxes = [
        [0.50, 0.50, 0.30, 0.35],  # Ana kutu (yüksek güven)
        [0.52, 0.51, 0.28, 0.33],  # Çok benzer (suppress edilecek)
        [0.48, 0.52, 0.32, 0.36],  # Çok benzer (suppress edilecek)
        [0.80, 0.30, 0.25, 0.25],  # Farklı nesne (tutulacak)
        [0.82, 0.32, 0.22, 0.23],  # Farklı nesne ile örtüşme (suppress)
        [0.20, 0.70, 0.20, 0.30],  # Üçüncü nesne (tutulacak)
    ]
    scores = [0.92, 0.85, 0.78, 0.88, 0.65, 0.70]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor('#0d1117')
    
    box_colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6']
    
    # Sol: NMS öncesi
    ax1 = axes[0]
    ax1.set_facecolor('#1a2035')
    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
    ax1.invert_yaxis()
    ax1.set_aspect('equal')
    
    for i, (box, score) in enumerate(zip(boxes, scores)):
        rect = patches.Rectangle(
            (box[0]-box[2]/2, box[1]-box[3]/2), box[2], box[3],
            linewidth=2, edgecolor=box_colors[i], facecolor=box_colors[i], alpha=0.2
        )
        ax1.add_patch(rect)
        ax1.text(box[0], box[1]-box[3]/2 - 0.02, f'{score:.2f}',
                 ha='center', va='bottom', color=box_colors[i], fontsize=10, fontweight='bold')
    
    ax1.set_title(f'NMS Öncesi: {len(boxes)} Tahmin', color='white', fontsize=13)
    ax1.set_xlabel('X', fontsize=11); ax1.set_ylabel('Y', fontsize=11)
    ax1.grid(True, alpha=0.15)
    ax1.text(0.5, 0.02, '🔴 Çakışan kutular bir nesneyi çok kez sayıyor!',
             ha='center', transform=ax1.transAxes, color='#ff6b6b', fontsize=10)
    
    # NMS uygula
    keep_idx, steps = non_maximum_suppression(boxes, scores, iou_threshold=0.5)
    
    # Sağ: NMS sonrası
    ax2 = axes[1]
    ax2.set_facecolor('#1a2035')
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.invert_yaxis()
    ax2.set_aspect('equal')
    
    suppressed_idx = set(range(len(boxes))) - set(keep_idx)
    
    for i, (box, score) in enumerate(zip(boxes, scores)):
        if i in suppressed_idx:
            rect = patches.Rectangle(
                (box[0]-box[2]/2, box[1]-box[3]/2), box[2], box[3],
                linewidth=1, edgecolor='#555555', facecolor='#555555', alpha=0.15,
                linestyle='--'
            )
            ax2.add_patch(rect)
            ax2.text(box[0], box[1], '✗', ha='center', va='center',
                     color='#555555', fontsize=16)
        else:
            rect = patches.Rectangle(
                (box[0]-box[2]/2, box[1]-box[3]/2), box[2], box[3],
                linewidth=3, edgecolor=box_colors[i], facecolor=box_colors[i], alpha=0.35
            )
            ax2.add_patch(rect)
            ax2.text(box[0], box[1]-box[3]/2 - 0.02, f'{score:.2f} ✓',
                     ha='center', va='bottom', color=box_colors[i], fontsize=10, fontweight='bold')
    
    ax2.set_title(f'NMS Sonrası: {len(keep_idx)} Tespit (IoU eşiği=0.5)', color='white', fontsize=13)
    ax2.set_xlabel('X', fontsize=11); ax2.set_ylabel('Y', fontsize=11)
    ax2.grid(True, alpha=0.15)
    ax2.text(0.5, 0.02, '✅ Her nesne için tek en iyi kutu kaldı!',
             ha='center', transform=ax2.transAxes, color='#2ecc71', fontsize=10)
    
    plt.suptitle('Non-Maximum Suppression (NMS)', color='white', fontsize=15, fontweight='bold')
    plt.tight_layout(pad=2)
    plt.show()
    
    print('\n📊 NMS Sonuçları:')
    print(f'   Başlangıç tahmin sayısı : {len(boxes)}')
    print(f'   Sonraki tespit sayısı    : {len(keep_idx)}')
    print(f'   Silinen (suppressed)     : {len(boxes)-len(keep_idx)}')
    print(f'   Tutulan indeksler        : {keep_idx}')

visualize_nms()


<a id='7'></a>
## 7. 🔮 Confidence Score & Tahmin Decode Etme

### YOLO Çıktısını Gerçek Koordinatlara Çevirme

Model, her grid cell için **göreli koordinatlar** üretir. Bunları gerçek piksel koordinatlarına çevirmek gerekir:

```
x_real = (j + x_offset) × (W / S)    # j = sütun indeksi
y_real = (i + y_offset) × (H / S)    # i = satır indeksi  
w_real = w_pred × W                   # Görüntü genişliğiyle ölçekle
h_real = h_pred × H                   # Görüntü yüksekliğiyle ölçekle
```


In [ ]:
def decode_predictions(output, S=7, B=2, C=20, 
                        img_w=448, img_h=448,
                        conf_threshold=0.3,
                        iou_threshold=0.5):
    """
    YOLO çıktı tensörünü (S×S×(B*5+C)) gerçek bounding box listesine dönüştürür
    
    Returns:
        List of dicts: [{box:[x1,y1,x2,y2], confidence, class_id, class_prob}, ...]
    """
    detections = []
    cell_w = img_w / S
    cell_h = img_h / S
    
    for row in range(S):
        for col in range(S):
            cell = output[row, col]
            class_probs = cell[B*5:]  # Son C eleman
            best_class  = np.argmax(class_probs)
            best_class_prob = class_probs[best_class]
            
            for b in range(B):
                offset = b * 5
                x_off  = cell[offset]      # [0,1] — cell içi offset
                y_off  = cell[offset + 1]  # [0,1] — cell içi offset
                w_norm = cell[offset + 2]  # [0,1] — görüntü genişliğine göre
                h_norm = cell[offset + 3]  # [0,1] — görüntü yüksekliğine göre
                conf   = cell[offset + 4]  # Güven skoru
                
                # Class confidence
                class_conf = conf * best_class_prob
                
                if class_conf < conf_threshold:
                    continue
                
                # Gerçek koordinatlara çevir
                x_center = (col + x_off) * cell_w
                y_center = (row + y_off) * cell_h
                width    = w_norm * img_w
                height   = h_norm * img_h
                
                # Köşe formatına çevir
                x1 = x_center - width  / 2
                y1 = y_center - height / 2
                x2 = x_center + width  / 2
                y2 = y_center + height / 2
                
                detections.append({
                    'box':        [x1, y1, x2, y2],
                    'box_center': [x_center, y_center, width, height],
                    'confidence': float(conf),
                    'class_id':   int(best_class),
                    'class_conf': float(class_conf),
                    'grid_cell':  (row, col),
                })
    
    return detections


# Demo: Dummy YOLO çıktısı üret ve decode et
np.random.seed(7)
S, B, C = 7, 2, 20
dummy_output = np.random.rand(S, S, B*5 + C) * 0.3

# Birkaç gerçekçi nesne ekle
# Nesne 1: Grid (2,3) — Kişi (class 14)
dummy_output[2, 3, 0:5]  = [0.4, 0.6, 0.3, 0.5, 0.92]   # Box 1
dummy_output[2, 3, 5:10] = [0.3, 0.5, 0.35, 0.5, 0.75]  # Box 2
dummy_output[2, 3, 10 + 14] = 0.95  # Class 14 = Person

# Nesne 2: Grid (5,1) — Araba (class 6)
dummy_output[5, 1, 0:5]  = [0.5, 0.4, 0.4, 0.3, 0.88]
dummy_output[5, 1, 10 + 6] = 0.91  # Class 6 = Car

detections = decode_predictions(dummy_output, conf_threshold=0.25)

VOC_CLASSES = ['aeroplane','bicycle','bird','boat','bottle',
               'bus','car','cat','chair','cow',
               'diningtable','dog','horse','motorbike','person',
               'pottedplant','sheep','sofa','train','tvmonitor']

print(f'🔍 Toplam {len(detections)} tespit (conf > 0.25):')
print(f'{"─"*60}')
for i, det in enumerate(sorted(detections, key=lambda x: -x['class_conf'])[:8]):
    print(f'  #{i+1} | Grid:{det["grid_cell"]} | '
          f'Sınıf: {VOC_CLASSES[det["class_id"]]:12s} | '
          f'Güven: {det["class_conf"]:.3f} | '
          f'Konum: ({det["box"][0]:.0f},{det["box"][1]:.0f})')


<a id='8'></a>
## 8. 🎬 Uçtan Uca YOLO Demo (Simülasyon)

Gerçek bir görüntü üzerinde **tam YOLO pipeline'ını** simüle edelim:
1. Görüntü → Grid'e böl
2. Tahminleri decode et
3. Confidence threshold uygula
4. NMS uygula
5. Final tespitleri göster


In [ ]:
class YOLOv1Simulator:
    """
    YOLOv1 pipeline simülatörü
    Gerçek sinir ağı yerine kontrollü dummy tahminler üretir
    """
    def __init__(self, S=7, B=2, C=20, img_size=448):
        self.S        = S
        self.B        = B
        self.C        = C
        self.img_size = img_size
        self.classes  = VOC_CLASSES
    
    def simulate_forward(self, gt_objects):
        """
        gt_objects: list of (class_id, cx, cy, w, h) — piksel cinsinden
        Gerçekçi YOLO çıktısı simüle eder (noise ile)
        """
        output = np.random.rand(self.S, self.S, self.B*5+self.C) * 0.05
        cell_size = self.img_size / self.S
        
        for (cls_id, cx, cy, w, h) in gt_objects:
            col = int(cx / cell_size)
            row = int(cy / cell_size)
            col = min(col, self.S - 1)
            row = min(row, self.S - 1)
            
            # Normalize
            x_off = (cx - col * cell_size) / cell_size
            y_off = (cy - row * cell_size) / cell_size
            w_norm = w / self.img_size
            h_norm = h / self.img_size
            
            noise = lambda: np.random.normal(0, 0.04)
            
            # Ana kutu — yüksek güven
            output[row, col, 0:5] = [
                np.clip(x_off + noise(), 0, 1),
                np.clip(y_off + noise(), 0, 1),
                np.clip(w_norm + noise(), 0.05, 1),
                np.clip(h_norm + noise(), 0.05, 1),
                0.88 + np.random.rand() * 0.1
            ]
            # İkinci kutu — biraz daha gürültülü (NMS ile silinecek)
            output[row, col, 5:10] = [
                np.clip(x_off + noise()*2, 0, 1),
                np.clip(y_off + noise()*2, 0, 1),
                np.clip(w_norm + noise()*2, 0.05, 1),
                np.clip(h_norm + noise()*2, 0.05, 1),
                0.65 + np.random.rand() * 0.1
            ]
            # Sınıf olasılıkları
            output[row, col, self.B*5 + cls_id] = 0.92 + np.random.rand() * 0.07
        
        return output
    
    def run_pipeline(self, gt_objects, conf_threshold=0.3, iou_threshold=0.45):
        """Tam inference pipeline"""
        # 1. Forward pass
        output = self.simulate_forward(gt_objects)
        
        # 2. Decode
        all_dets = decode_predictions(
            output, self.S, self.B, self.C,
            self.img_size, self.img_size, conf_threshold
        )
        
        # 3. Per-class NMS
        final_dets = []
        for cls_id in range(self.C):
            cls_dets = [d for d in all_dets if d['class_id'] == cls_id]
            if not cls_dets:
                continue
            boxes_c  = [d['box_center'] for d in cls_dets]
            scores_c = [d['class_conf'] for d in cls_dets]
            keep, _  = non_maximum_suppression(boxes_c, scores_c, iou_threshold)
            final_dets.extend([cls_dets[k] for k in keep])
        
        return output, all_dets, final_dets
    
    def visualize_results(self, gt_objects, final_dets):
        """Sonuçları görselleştir"""
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        fig.patch.set_facecolor('#0d1117')
        
        cls_colors = plt.cm.Set2(np.linspace(0, 1, self.C))
        cell_size  = self.img_size / self.S
        
        for ax_idx, ax in enumerate(axes):
            # Arka plan görüntüsü (simüle)
            bg = np.ones((self.img_size, self.img_size, 3)) * 0.1
            ax.imshow(bg, extent=[0, self.img_size, self.img_size, 0])
            ax.set_facecolor('#0d1117')
            ax.set_xlim(0, self.img_size)
            ax.set_ylim(self.img_size, 0)
            
            # Grid
            for k in range(self.S + 1):
                ax.axhline(k * cell_size, color='#333355', lw=0.5, alpha=0.6)
                ax.axvline(k * cell_size, color='#333355', lw=0.5, alpha=0.6)
            
            # Ground truth
            for (cls_id, cx, cy, w, h) in gt_objects:
                gt_rect = patches.Rectangle(
                    (cx-w/2, cy-h/2), w, h,
                    linewidth=2, edgecolor='lime', facecolor='none',
                    linestyle='--', label='Ground Truth'
                )
                ax.add_patch(gt_rect)
                ax.text(cx-w/2, cy-h/2 - 4, f'GT: {self.classes[cls_id]}',
                        color='lime', fontsize=8, fontweight='bold')
            
            if ax_idx == 1:
                # Tahminler
                for det in final_dets:
                    x1, y1, x2, y2 = det['box']
                    c = det['class_id']
                    color = cls_colors[c % len(cls_colors)]
                    pred_rect = patches.Rectangle(
                        (x1, y1), x2-x1, y2-y1,
                        linewidth=2.5, edgecolor=color, facecolor=color,
                        alpha=0.25
                    )
                    ax.add_patch(pred_rect)
                    label = f'{self.classes[c]} {det["class_conf"]:.2f}'
                    ax.text(x1, y1 - 3, label, color='white',
                            fontsize=8, fontweight='bold',
                            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor='none'))
            
            title = ('Ground Truth' if ax_idx == 0 
                     else f'YOLOv1 Tahminleri ({len(final_dets)} tespit)')
            ax.set_title(title, color='white', fontsize=13, fontweight='bold')
            ax.set_xlabel('X (piksel)'); ax.set_ylabel('Y (piksel)')
        
        plt.suptitle('YOLOv1 Simülasyonu — Uçtan Uca Pipeline', 
                     color='white', fontsize=14, fontweight='bold')
        plt.tight_layout(pad=2)
        plt.show()


# Çalıştır!
np.random.seed(42)
sim = YOLOv1Simulator()

# Simüle edilmiş GT: (class_id, cx, cy, w, h) piksel cinsinden
gt_objects = [
    (14, 150, 200, 120, 160),  # Person
    ( 6, 320, 280, 140, 100),  # Car
    ( 1, 380, 100,  80,  80),  # Bicycle
]

output, all_dets, final_dets = sim.run_pipeline(gt_objects, conf_threshold=0.3)
sim.visualize_results(gt_objects, final_dets)

print(f'\n📊 Pipeline Özeti:')
print(f'   Girdi grid hücresi sayısı  : {sim.S*sim.S} ({sim.S}×{sim.S})')
print(f'   Ham tahmin sayısı          : {sim.S*sim.S*sim.B}')
print(f'   Threshold sonrası          : {len(all_dets)}')
print(f'   NMS sonrası (final)        : {len(final_dets)}')


<a id='9'></a>
## 9. 📊 Performans Analizi & Kısıtlamalar

### mAP (Mean Average Precision)

$$\text{AP} = \int_0^1 P(R) \, dR$$
$$\text{mAP} = \frac{1}{C} \sum_{c=1}^C \text{AP}_c$$

### Pascal VOC 2007 Karşılaştırması

| Model | mAP | FPS | Tip |
|-------|-----|-----|-----|
| R-CNN | 66.0 | 0.05 | Two-stage |
| Fast R-CNN | 70.0 | 0.5 | Two-stage |
| Faster R-CNN | 73.2 | 7 | Two-stage |
| **YOLOv1** | **63.4** | **45** | **One-stage** |
| Fast-YOLO | 52.7 | 155 | One-stage |

### YOLOv1'in Kısıtlamaları
1. **Her grid cell yalnızca 1 nesne** tespit edebilir → kümelerdeki nesneleri kaçırır
2. **Küçük nesnelerde** zayıf performans
3. **Yeni aspect ratio'larda** düşük doğruluk
4. **Loss'taki oran dengesizliği** — hata büyük/küçük kutularda eşit değil


In [ ]:
def plot_performance_comparison():
    """Nesne tespit modeli karşılaştırma grafikleri"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor('#0d1117')
    
    models  = ['R-CNN', 'Fast\nR-CNN', 'Faster\nR-CNN', 'YOLOv1', 'Fast\nYOLO']
    mAP     = [66.0,     70.0,          73.2,             63.4,     52.7]
    fps     = [0.05,      0.5,            7.0,             45.0,    155.0]
    colors  = ['#95a5a6', '#7f8c8d', '#bdc3c7', '#e74c3c', '#e67e22']
    
    # 1. mAP karşılaştırması
    ax1 = axes[0]
    bars = ax1.barh(models, mAP, color=colors, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, mAP):
        ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val}%', va='center', color='white', fontsize=10, fontweight='bold')
    ax1.axvline(x=63.4, color='#e74c3c', linestyle='--', alpha=0.5, linewidth=1.5)
    ax1.set_xlim(40, 80)
    ax1.set_title('mAP @ VOC 2007', color='white', fontsize=12)
    ax1.set_xlabel('mAP (%)', fontsize=11)
    
    # 2. FPS karşılaştırması (log scale)
    ax2 = axes[1]
    bars2 = ax2.barh(models, fps, color=colors, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars2, fps):
        ax2.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2,
                 f'{val} fps', va='center', color='white', fontsize=10, fontweight='bold')
    ax2.set_xscale('log')
    ax2.axvline(x=30, color='cyan', linestyle='--', alpha=0.6, linewidth=1.5,
                label='Gerçek zamanlı eşiği (30fps)')
    ax2.set_title('Hız (FPS, log ölçek)', color='white', fontsize=12)
    ax2.set_xlabel('Frames per Second', fontsize=11)
    ax2.legend(fontsize=9)
    
    # 3. mAP vs FPS scatter (speed-accuracy tradeoff)
    ax3 = axes[2]
    scatter_colors = colors
    for i, (m, f, name, c) in enumerate(zip(mAP, fps, models, scatter_colors)):
        ax3.scatter(f, m, s=200, c=c, edgecolors='white', linewidth=1.5, zorder=5)
        ax3.annotate(name.replace('\n', ' '), (f, m),
                     textcoords='offset points', xytext=(8, 4),
                     fontsize=9, color='white')
    
    ax3.axvline(x=30, color='cyan', linestyle='--', alpha=0.5, linewidth=1.5)
    ax3.axhline(y=63.4, color='#e74c3c', linestyle='--', alpha=0.5, linewidth=1.5)
    ax3.set_xscale('log')
    ax3.set_xlabel('Hız (FPS, log)', fontsize=11)
    ax3.set_ylabel('mAP (%)', fontsize=11)
    ax3.set_title('Hız-Doğruluk Dengesi', color='white', fontsize=12)
    ax3.set_xlim(0.01, 300)
    ax3.set_ylim(45, 80)
    
    # YOLO bölgesini vurgula
    ax3.axvspan(30, 300, alpha=0.05, color='green', label='Gerçek Zamanlı Bölge')
    ax3.legend(fontsize=9)
    
    plt.suptitle('Nesne Tespit Modelleri — Performans Karşılaştırması',
                 color='white', fontsize=14, fontweight='bold')
    plt.tight_layout(pad=2)
    plt.show()

plot_performance_comparison()

# Precision-Recall eğrisi simülasyonu
def plot_precision_recall():
    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')
    
    # Simüle edilmiş PR eğrileri
    recall = np.linspace(0, 1, 100)
    
    def pr_curve(quality):
        return quality * (1 - recall**1.5) + (1-quality) * (0.5 - recall*0.3)
    
    for cls, color, q in [
        ('person',  '#e74c3c', 0.82),
        ('car',     '#3498db', 0.78),
        ('bicycle', '#2ecc71', 0.71),
        ('dog',     '#f39c12', 0.65),
    ]:
        prec = np.clip(pr_curve(q), 0, 1)
        ap   = np.trapz(prec, recall)
        ax.plot(recall, prec, color=color, linewidth=2.2,
                label=f'{cls} (AP={ap:.2f})')
        ax.fill_between(recall, prec, alpha=0.08, color=color)
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Eğrisi (Simülasyon)', color='white', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_precision_recall()


<a id='10'></a>
## 10. 🎓 Özet & Sonraki Adımlar

### YOLOv1'in Temel Katkıları

| Katkı | Açıklama | Önemi |
|-------|----------|-------|
| **Tek regresyon** | Tespit = regresyon problemi | Uçtan uca öğrenilebilir |
| **Grid-based** | S×S grid hücre sorumluluğu | Verimli paralel işleme |
| **Unified model** | Tek forward pass | 45 FPS gerçek zamanlı |
| **End-to-end** | Tek kayıp fonksiyonu | Optimize edilebilirlik |

### YOLO Serisi Evrim
```
YOLOv1 (2016) → YOLOv2/YOLO9000 (2017) → YOLOv3 (2018)
    ↓                    ↓                       ↓
Temel fikir        Anchor boxes,          Multi-scale,
Grid-based         BatchNorm,             Darknet-53,
                   Multi-label            FPN benzeri
```

### Bu Notebook'ta Öğrendiklerimiz
- ✅ YOLO'nun grid tabanlı tespit mantığı
- ✅ 7×7×30 çıktı tensörü yapısı
- ✅ 5 bileşenli loss fonksiyonu (λ_coord=5, λ_noobj=0.5)
- ✅ IoU hesabı ve NMS algoritması
- ✅ Tahmin decode etme (göreli → piksel koordinatları)
- ✅ Uçtan uca inference pipeline

### 🚀 Sonraki Adımlar
1. **YOLOv2'ye geçiş:** Anchor box kavramını incele
2. **PyTorch implementasyonu:** Gerçek Darknet modelini kodla
3. **COCO/VOC dataseti:** Gerçek veri ile eğit
4. **mAP hesabı:** Gerçek değerlendirme metriğini implement et

---
> 📖 **Referans:** Redmon et al. *"You Only Look Once: Unified, Real-Time Object Detection"* CVPR 2016  
> 🔗 https://arxiv.org/abs/1506.02640


In [ ]:
# ── YOLOv1 Konsept Özet Visualizasyonu ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, 16); ax.set_ylim(0, 8)
ax.axis('off')

# Başlık
ax.text(8, 7.5, 'YOLOv1 — Öğrendiklerimiz', ha='center', va='center',
        color='white', fontsize=18, fontweight='bold')

concepts = [
    (2,  5.5, '#e74c3c', '🗺️ Grid', 'S×S=7×7 hücre\nHer hücre B=2 kutu\ntahmin eder'),
    (5.5,5.5, '#3498db', '📦 Tensör', 'S×S×(B×5+C)\n= 7×7×30\nÇıktı boyutu'),
    (9,  5.5, '#2ecc71', '📐 Loss', '5 bileşen\nλ_coord=5\nλ_noobj=0.5'),
    (12.5,5.5,'#f39c12', '🔢 IoU',  'Kesişim/Birleşim\n0.5 eşiği\nDoğruluk ölçüsü'),
    (2,  2.5, '#9b59b6', '🔧 NMS',  'Tekrarlı kutuları\neliminer\nIoU eşiği ile'),
    (5.5,2.5, '#1abc9c', '⚡ Hız',  '45 FPS base\n155 FPS fast\nGerçek zamanlı!'),
    (9,  2.5, '#e67e22', '🎯 mAP',  'VOC2007: 63.4%\nHız-doğruluk\ndengesi'),
    (12.5,2.5,'#c0392b', '⚠️ Limit','7×7=49 nesne max\nKüçük nesneler\nzayıf'),
]

for cx, cy, color, title, desc in concepts:
    box = FancyBboxPatch((cx-1.6, cy-1.2), 3.2, 2.4,
                          boxstyle='round,pad=0.15',
                          facecolor=color, alpha=0.2,
                          edgecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(cx, cy + 0.7, title, ha='center', va='center',
            color=color, fontsize=12, fontweight='bold')
    ax.text(cx, cy - 0.2, desc, ha='center', va='center',
            color='#cccccc', fontsize=9)

ax.text(8, 0.4, '"You Only Look Once" — Tek bakışta nesne tespiti devrim niteliğindeydi! 🚀',
        ha='center', va='center', color='#aaaaaa', fontsize=11, style='italic')

plt.tight_layout()
plt.show()
print('\n✅ YOLOv1 Notebook tamamlandı!')
print('📌 Sonraki: YOLOv2 — Anchor Box, BatchNorm ve YOLO9000 (9000 sınıf!)')
